In [ ]:
class EvalState(TypedDict):
    question: str
    answer: str
    feedback: str
    is_valid: bool
    attempts: int


In [ ]:
generator_prompt = ChatPromptTemplate.from_messages([
    ("system", "You answer the question clearly and correctly."),
    ("human", """
Question:
{question}

Previous answer:
{answer}

Evaluator feedback:
{feedback}

If feedback is present, fix the issues and improve the answer.
""")
])


In [ ]:
from langchain_openai import ChatOpenAI
import json

llm = ChatOpenAI(model="gpt-4o-mini")


In [ ]:
def generator_node(state: EvalState):
    response = llm.invoke(
        generator_prompt.format(
            question=state["question"],
            answer=state.get("answer", ""),
            feedback=state.get("feedback", "")
        )
    )
    return {
        "answer": response.content,
        "attempts": state["attempts"] + 1
    }


In [ ]:
def evaluator_node(state: EvalState):
    response = llm.invoke(
        evaluator_prompt.format(
            question=state["question"],
            answer=state["answer"]
        )
    )

    result = json.loads(response.content)
    return {
        "is_valid": result["is_valid"],
        "feedback": result["feedback"]
    }


In [ ]:
def should_retry(state: EvalState):
    if state["is_valid"]:
        return "end"
    if state["attempts"] >= 3:
        return "end"
    return "retry"


In [ ]:
from langgraph.graph import StateGraph, END

graph = StateGraph(EvalState)

graph.add_node("generate", generator_node)
graph.add_node("evaluate", evaluator_node)

graph.set_entry_point("generate")
graph.add_edge("generate", "evaluate")

graph.add_conditional_edges(
    "evaluate",
    should_retry,
    {
        "retry": "generate",
        "end": END
    }
)

app = graph.compile()


In [ ]:
result = app.invoke({
    "question": "Explain Kubernetes in one paragraph",
    "answer": "",
    "feedback": "",
    "is_valid": False,
    "attempts": 0
})

print("Final answer:\n", result["answer"])
print("Attempts:", result["attempts"])
print("Valid:", result["is_valid"])
